# 004 Add a Script to a Skill

这是第四课：什么时候该给 Skill 增加 `scripts/`。

学习目标：

1. 理解什么时候一个动作已经值得从文字说明升级为脚本
2. 学会判断“workflow”与“deterministic transformation”的边界
3. 结合真实天气 Skill，新增第一个脚本 `normalize_location.py`
4. 理解 Skill 的成长顺序为什么通常是：最小版 -> metadata -> references -> scripts

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

到第三课为止，这份天气 Skill 已经有了：

- `SKILL.md`
- `agents/openai.yaml`
- `references/weather_sources.md`

但现在还有一个动作开始显得重复，而且适合确定性处理：

- 地点标准化

例如：

- `beijing`
- ` New   York `
- `jfk`
- `los angeles, ca`

这些输入如果每次都靠自然语言临时处理，稳定性会越来越差。

这时就值得引入第一个 `scripts/`。


## 先看重构后的真实目录

第四课不是新造一个抽象示例，而是继续扩展现有真实 Skill。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   └── normalize_location.py
└── SKILL.md


## 先看 `SKILL.md` 的变化

你应该观察到两件事：

1. workflow 里增加了“Normalize the location”
2. 下面新增了 `Scripts` 小节

这说明脚本不是孤立存在的，它必须被 Skill 的 workflow 明确引用。


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Use `wttr.in` as the primary source.
5. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
6. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
7. Do not guess when weather data is unavailable.

## References

- Read `references/weather_sources.md` for concrete `wttr.in` and Open-Meteo query patterns.

## 为什么这一步值得脚本化

不是所有动作都值得写脚本。

但地点标准化满足几个很强的信号：

1. 会重复出现
2. 规则相对稳定
3. 结果应该可预测
4. 如果每次都临时写，容易前后不一致

这就是典型的 deterministic transformation。


In [4]:
why_script_now = {
    'repeated_action': True,
    'deterministic_rules': True,
    'small_input_output_boundary': True,
    'benefit_from_reuse': True,
}

from pprint import pprint
pprint(why_script_now)


{'benefit_from_reuse': True,
 'deterministic_rules': True,
 'repeated_action': True,
 'small_input_output_boundary': True}


## 读真实脚本 `normalize_location.py`

这里的目标不是做一个世界级地理标准化器。

第一版脚本只做小而稳定的处理：

- 去掉多余空格
- 把逗号转成空格
- 把城市名标准化
- 把空格变成 `+`，便于 `wttr.in` URL 使用
- 对少量机场代码保留大写


In [5]:
print((skill_root / 'scripts' / 'normalize_location.py').read_text(encoding='utf-8'))


#!/usr/bin/env python3
import re
import sys


def normalize_location(raw: str) -> str:
    value = raw.strip()
    value = re.sub(r"\s+", " ", value)
    value = value.replace(",", " ")
    value = re.sub(r"\s+", " ", value).strip()

    upper_special = {"jfk", "lax", "sfo", "lhr", "cdg", "hnd"}
    if value.lower() in upper_special:
        return value.upper()

    parts = []
    for token in value.split(" "):
        if len(token) <= 3 and token.isalpha() and token.isupper():
            parts.append(token)
        elif token.isalpha():
            parts.append(token.capitalize())
        else:
            parts.append(token)

    return "+".join(parts)


def main() -> int:
    if len(sys.argv) != 2:
        print("Usage: normalize_location.py <location>")
        return 1

    print(normalize_location(sys.argv[1]))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())



## 直接跑几个例子

正式开发时，不要只读脚本，要真的跑一下。


In [6]:
import subprocess

samples = [
    'beijing',
    ' New   York ',
    'jfk',
    'los angeles, ca',
]

for sample in samples:
    result = subprocess.run(
        [
            'python',
            '.agents/skills/weather-query-assistant/scripts/normalize_location.py',
            sample,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    print(sample, '->', result.stdout.strip(), '| returncode =', result.returncode)


beijing -> Beijing | returncode = 0
 New   York  -> New+York | returncode = 0
jfk -> JFK | returncode = 0
los angeles, ca -> Los+Angeles+Ca | returncode = 0


## 这一步和 `references/` 的区别

很多人学到这里会把 `references/` 和 `scripts/` 混起来。

你可以这样强行区分：

- `references/`：告诉你“怎么查、有哪些资料”
- `scripts/`：直接替你做一个稳定的小动作

在天气 Skill 里：

- `weather_sources.md` 是资料
- `normalize_location.py` 是动作


In [7]:
references_vs_scripts = {
    'references/': 'lookup material and examples',
    'scripts/': 'repeatable deterministic action',
}

pprint(references_vs_scripts)


{'references/': 'lookup material and examples',
 'scripts/': 'repeatable deterministic action'}


## 什么时候还不该写脚本

也不是所有规则都要立刻脚本化。

下面这些情况仍然不建议急着写脚本：

- 规则还在频繁变化
- 你还没搞清楚输入输出边界
- 只是偶尔一次性动作
- 用文字说明已经足够清楚


In [8]:
when_not_to_script = [
    'rules still changing quickly',
    'input/output boundary is still unclear',
    'one-off action',
    'plain instructions are still enough',
]

pprint(when_not_to_script)


['rules still changing quickly',
 'input/output boundary is still unclear',
 'one-off action',
 'plain instructions are still enough']


## 正式开发时，一份 Skill 加脚本的稳妥顺序

这里给你一条很实用的顺序。

1. 先把 workflow 写清楚
2. 再观察哪里开始重复
3. 找出最小的、稳定的动作边界
4. 只把这一小段抽成脚本
5. 在 `SKILL.md` 里明确写出何时运行这个脚本

不要跳过中间步骤，一上来就造“大工具箱”。


In [9]:
script_adoption_flow = [
    'workflow first',
    'observe repetition',
    'find deterministic boundary',
    'extract smallest useful script',
    'wire it back into SKILL.md',
]

pprint(script_adoption_flow)


['workflow first',
 'observe repetition',
 'find deterministic boundary',
 'extract smallest useful script',
 'wire it back into SKILL.md']


## 这节课的正式开发价值

到第四课，你已经看到一份 Skill 的真实成长路径：

1. 最小 Skill
2. 加 UI metadata
3. 加 references
4. 加 scripts

这比一开始就堆一个“大而全目录树”更符合真实开发节奏。


## 如果继续往下走，第五课最自然的方向是什么

现在脚本已经有了，下一步最自然不是再加更多脚本，而是：

- 把脚本真正接入一个完整的使用流程

例如下一课可以做：

1. 先标准化地点
2. 再拼出 `wttr.in` 查询 URL
3. 再返回适合展示的最终查询字符串

这样你就会看到一个 Skill 里的多个部分开始协同工作。


In [10]:
lesson_five_candidate = {
    'theme': 'wire the weather skill into a complete mini workflow',
    'steps': [
        'normalize location',
        'build wttr.in query',
        'return a final query-ready string or command',
    ],
}

pprint(lesson_five_candidate)


{'steps': ['normalize location',
           'build wttr.in query',
           'return a final query-ready string or command'],
 'theme': 'wire the weather skill into a complete mini workflow'}


## 当前阶段结论

你现在需要记住：

1. `scripts/` 适合承载重复且确定性强的小动作
2. 不是所有逻辑都该脚本化，只有边界稳定时才值得抽出来
3. 脚本不能脱离 Skill 单独存在，应该在 `SKILL.md` 里明确说明何时运行
4. 这份天气 Skill 的 `normalize_location.py` 是一个很好的第一脚本例子
5. Skill 的成长顺序通常是：最小版 -> metadata -> references -> scripts -> workflow integration

下一步建议：

- 继续第五课：把天气 Skill 的几个部分接成一个完整小流程
